# 1. Proposito de la demo extrema

Este notebook construye un caso deliberadamente complejo para forzar el analizador de dependencias de Inspyro: varios archivos `.py`, imports relativos, reexports, clases, propiedades, atributos `self`, funciones encadenadas y un reporte DOCX final. La historia visible se mantiene corta; la complejidad vive en `demo_dependency_extreme/`.

## 1.1. Preparar entorno e importar la fachada publica

La primera celda deja el workspace en `sys.path`, importa la fachada del paquete y muestra los simbolos que conviene analizar con `analyze_dependencies` y `analyze_impact`.

In [1]:
from pathlib import Path
import sys

WORKSPACE = Path.cwd()
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from demo_dependency_extreme import (
    EXPECTED_DEPENDENCY_TARGETS,
    EXPECTED_IMPACT_TARGETS,
    Section,
    BeamModel,
    build_demo_model,
    run_demo_scenario,
    stage_00,
    stage_45,
    target_rows,
)

print('Workspace:', WORKSPACE)
print('Paquete de estres:', WORKSPACE / 'demo_dependency_extreme')
print('Targets dependencias:', ', '.join(EXPECTED_DEPENDENCY_TARGETS))
print('Targets impacto:', ', '.join(EXPECTED_IMPACT_TARGETS))

Workspace: C:\CalcPyro\P1
Paquete de estres: C:\CalcPyro\P1\demo_dependency_extreme
Targets dependencias: final_utilization, BeamModel.capacity_ratio, Section.area, stage_45
Targets impacto: base_width, steel_fy, LoadCase.dead, stage_00


## 2. Ejecutar el escenario base

El escenario arma material, geometria, cargas, seccion, modelo y verificaciones. La variable `final_utilization` queda visible en el notebook para analizar el enlace notebook -> paquete -> clases -> cadena profunda.

In [2]:
scenario = run_demo_scenario()
model = scenario['model']
checks = scenario['checks']
final_utilization = scenario['utilization']

print(scenario['summary'])
print('Tipo de modelo:', type(model).__name__)
print('Tipo de seccion:', type(model.section).__name__)
print('Razon gobernante final_utilization:', round(final_utilization, 6))

try:
    import pandas as pd
    from IPython.display import display
    display(pd.DataFrame(scenario['audit_rows'], columns=['Magnitud publica', 'Valor', 'Unidad']))
    display(pd.DataFrame(scenario['check_rows'], columns=['Verificacion', 'Razon', 'Limite', 'Estado', 'Simbolo fuente']))
except Exception:
    print('Filas de auditoria:', scenario['audit_rows'])
    print('Filas de verificaciones:', scenario['check_rows'])

Utilizacion gobernante: 0.8807
Momento amplificado: 4414.8199 kN m
Cadena profunda stage_45: 151.601660
Tipo de modelo: BeamModel
Tipo de seccion: Section
Razon gobernante final_utilization: 0.880702


,Magnitud publica,Valor,Unidad
0,Carga lineal equivalente,107.380000,kN/m
1,Momento de servicio,4348.890000,kN m
2,Momento amplificado,4414.819900,kN m
3,Razon demanda/capacidad,0.838973,-
4,Cadena etapa 25,134.137848,-
5,Cadena etapa 45,151.601660,-


,Verificacion,Razon,Limite,Estado,Simbolo fuente
0,Flexure envelope,0.838973,1.00,OK,BeamModel.capacity_ratio
1,Static chain depth,1.325354,1.75,OK,stage_45
2,Geometry/material coupling,0.880702,1.00,OK,base_width -> steel_fy -> Section.area


## 3. Revisar la cadena transitiva de 46 etapas

`stage_45` llama de forma estatica a `stage_44`, que depende de `stage_43`, y asi hasta `stage_00`. Esa cadena mezcla constantes de `geometry.py` y `materials.py`, por lo que sirve para probar `max_depth` y truncamiento.

In [3]:
from demo_dependency_extreme.chain import chain_summary, stage_05, stage_25, ultimate_stage

chain = chain_summary()
print('stage_00:', round(stage_00(), 6))
print('stage_05:', round(stage_05(), 6))
print('stage_25:', round(stage_25(), 6))
print('stage_45:', round(stage_45(), 6))
print('ultimate_stage alias:', round(ultimate_stage(), 6))
print('Resumen cadena:', chain)

stage_00: 114.385745
stage_05: 118.211589
stage_25: 134.137848
stage_45: 151.60166
ultimate_stage alias: 151.60166
Resumen cadena: {'stage_05': 118.21158929157409, 'stage_25': 134.137847517223, 'stage_45': 151.6016601918721}


## 4. Ejercitar clases, instancias, propiedades y atributos self

Esta celda toca `Section.area`, `Section.inertia`, `BeamModel.capacity_ratio` y atributos de instancia de `LoadCase`. Es el recorrido que debe obligar al analizador a distinguir metodos, propiedades, constructores y `self.attr`.

In [4]:
section = model.section
load_case = model.load_case

section_area = section.area
section_inertia = section.inertia
capacity_ratio = model.capacity_ratio()
load_dead = load_case.dead
load_service = load_case.service_total

print('Section.area:', round(section_area, 6))
print('Section.inertia:', round(section_inertia, 9))
print('BeamModel.capacity_ratio:', round(capacity_ratio, 6))
print('LoadCase.dead:', round(load_dead, 6))
print('LoadCase.service_total:', round(load_service, 6))
print('MRO Section:', [cls.__name__ for cls in type(section).mro()])

Section.area: 0.028664
Section.inertia: 0.00245166
BeamModel.capacity_ratio: 0.838973
LoadCase.dead: 756.0
LoadCase.service_total: 1469.88
MRO Section: ['Section', 'object']


## 5. Registrar targets de estres y casos ambiguos esperados

El paquete contiene dos funciones `duplicate_name`, una en `materials.py` y otra en `geometry.py`. Ese homonimo es intencional: debe degradar de forma conservadora si el analizador no tiene contexto suficiente.

In [5]:
from demo_dependency_extreme.materials import duplicate_name as material_duplicate_name
from demo_dependency_extreme.geometry import duplicate_name as geometry_duplicate_name

stress_targets = target_rows()
ambiguous_probe_material = material_duplicate_name(2.0)
ambiguous_probe_geometry = geometry_duplicate_name(2.0)
expected_unresolved_case = 'getattr(model, dynamic_name) no pertenece al camino principal'

print('Targets recomendados:')
for mode, symbol, expected_path in stress_targets:
    print(f'- {mode}: {symbol} -> {expected_path}')
print('Homonomo material:', round(ambiguous_probe_material, 6))
print('Homonomo geometria:', round(ambiguous_probe_geometry, 6))
print('Caso dinamico documentado:', expected_unresolved_case)

Targets recomendados:
- Dependencias: final_utilization -> Notebook/checks/model/sections/materials/loads/chain
- Dependencias: BeamModel.capacity_ratio -> model -> loads -> sections -> materials -> chain
- Dependencias: Section.area -> sections -> geometry
- Dependencias: stage_45 -> chain stage_45 ... stage_00 -> geometry/materials
- Impacto: base_width -> geometry -> sections -> model -> checks -> notebook
- Impacto: steel_fy -> materials -> chain/sections/checks -> notebook
- Impacto: LoadCase.dead -> loads -> model -> checks -> notebook
- Impacto: stage_00 -> chain -> stage_45 -> model/checks -> notebook
Homonomo material: 2.355
Homonomo geometria: 0.64
Caso dinamico documentado: getattr(model, dynamic_name) no pertenece al camino principal


## 6. Generar reporte DOCX del caso de prueba

El reporte usa bloques estables, tablas con caption, ecuaciones en LaTeX y una revision compacta con `doc_finalize(profile="delivery")` cuando el kernel la expone. La auditoria MCP externa cierra el flujo con `check_document_quality`.

In [6]:
doc_reset(hard=True)

with build_doc(block_id='extreme_cover', order=10) as builder:
    builder.metadata(
        title='Demo extrema del analizador de dependencias',
        subject='Stress test estatico con notebook, modulos Python y DOCX',
        keywords=['inspyro', 'dependencias', 'impacto', 'docx'],
    )
    builder.heading('Demo extrema del analizador de dependencias', level=1)
    builder.text('Se construye un caso de prueba estatico para recorrer dependencias hacia atras e impacto hacia delante a traves de notebook, paquete Python, clases, funciones, propiedades y una cadena profunda de calculo.')
    builder.table(
        stress_targets,
        headers=['Modo', 'Simbolo', 'Ruta esperada'],
        caption='Targets recomendados para el analizador',
        label='tbl:targets-extreme',
    )
    builder.text('Fuente: elaboracion propia.')

with build_doc(block_id='extreme_equations', order=20) as builder:
    builder.heading('Ecuaciones de control', level=2)
    builder.text('La razon gobernante compara demanda y capacidad con una envolvente conservadora.')
    builder.math_latex(r'\eta = \max\left(\frac{M_u}{\phi M_n},\frac{V_u}{\phi V_n},\frac{\sigma_T}{F_y}\right)', label='eq:utilizacion-extrema', number=True)
    builder.text('La cadena profunda se representa como una composicion estatica de funciones puras.')
    builder.math_latex(r'S_{45}=f_{45}(f_{44}(\cdots f_{1}(f_{0}(S_0))\cdots))', label='eq:cadena-extrema', number=True)

with build_doc(block_id='extreme_results', order=30) as builder:
    builder.heading('Resultados del escenario base', level=2)
    builder.table(
        scenario['audit_rows'],
        headers=['Magnitud publica', 'Valor', 'Unidad'],
        caption='Resumen de auditoria del modelo',
        label='tbl:auditoria-extrema',
    )
    builder.text('La utilizacion final queda en el notebook como `final_utilization`, conectando el resultado visible con el grafo de dependencias.')
    builder.table(
        scenario['check_rows'],
        headers=['Verificacion', 'Razon', 'Limite', 'Estado', 'Simbolo fuente'],
        caption='Verificaciones que alimentan la utilizacion final',
        label='tbl:checks-extreme',
    )
    builder.text('Fuente: elaboracion propia.')

with build_doc(block_id='extreme_interpretation', order=40) as builder:
    builder.heading('Lectura esperada del analizador', level=2)
    builder.text('Con profundidad alta, el grafo debe atravesar facade.py, checks.py, model.py, sections.py, loads.py, materials.py, geometry.py y chain.py. Con profundidad baja, debe marcar truncamiento sin inventar dependencias ambiguas.')
    builder.list([
        'Dependencias hacia atras: final_utilization, BeamModel.capacity_ratio, Section.area y stage_45.',
        'Impacto hacia delante: base_width, steel_fy, LoadCase.dead y stage_00.',
        'Ambiguedad controlada: duplicate_name existe en dos modulos y debe resolverse solo con contexto suficiente.',
    ], ordered=False)

if 'doc_finalize' in globals():
    quality = doc_finalize(profile='delivery', detail='summary')
else:
    quality = {'status': 'deferred_to_mcp', 'score': None, 'counts': {}, 'note': 'Use MCP check_document_quality after DOCX export.'}

print('DOCX quality status:', quality.get('status'))
print('DOCX quality score:', quality.get('score'))
print('DOCX quality counts:', quality.get('counts'))

DOCX quality status: deferred_to_mcp
DOCX quality score: None
DOCX quality counts: {}


## 7. Payloads sugeridos para MCP analysis

Estas rutas quedan impresas para que el mismo notebook sirva como guia de prueba. Las llamadas reales a `analyze_dependencies` y `analyze_impact` se ejecutan desde MCP con perfil `analysis`, usando este archivo y el `kernel_id` activo.

In [7]:
analysis_payloads = {
    'dependencies_high_depth': {
        'symbol': 'final_utilization',
        'file_path': str(WORKSPACE / 'demo_dependency_analyzer_extreme.ipynb'),
        'max_depth': 80,
    },
    'dependencies_truncated': {
        'symbol': 'stage_45',
        'file_path': str(WORKSPACE / 'demo_dependency_extreme' / 'chain.py'),
        'max_depth': 5,
    },
    'impact_high_depth': {
        'symbol': 'base_width',
        'file_path': str(WORKSPACE / 'demo_dependency_extreme' / 'geometry.py'),
        'max_depth': 80,
    },
    'impact_truncated': {
        'symbol': 'stage_00',
        'file_path': str(WORKSPACE / 'demo_dependency_extreme' / 'chain.py'),
        'max_depth': 5,
    },
}
for name, payload in analysis_payloads.items():
    print(name, payload)

dependencies_high_depth {'symbol': 'final_utilization', 'file_path': 'C:\\CalcPyro\\P1\\demo_dependency_analyzer_extreme.ipynb', 'max_depth': 80}
dependencies_truncated {'symbol': 'stage_45', 'file_path': 'C:\\CalcPyro\\P1\\demo_dependency_extreme\\chain.py', 'max_depth': 5}
impact_high_depth {'symbol': 'base_width', 'file_path': 'C:\\CalcPyro\\P1\\demo_dependency_extreme\\geometry.py', 'max_depth': 80}
impact_truncated {'symbol': 'stage_00', 'file_path': 'C:\\CalcPyro\\P1\\demo_dependency_extreme\\chain.py', 'max_depth': 5}
